In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')
import joblib

In [7]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, col='avg_monthly_price'):
        self.col = col
    def fit(self, X, y=None):
        Q1 = X[self.col].quantile(0.25)
        Q3 = X[self.col].quantile(0.75)
        IQR = Q3 - Q1
        self.lower_ = Q1 - 1.5 * IQR
        self.upper_ = Q3 + 1.5 * IQR
        return self
    def transform(self, X):
        df = X.copy()
        df[self.col] = df[self.col].clip(lower=self.lower_, upper=self.upper_)
        return df

In [8]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, col='avg_monthly_price'):
        self.col = col
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        df = X.copy()
        df['month']     = df.index.month
        df['quarter']   = df.index.quarter
        df['year']      = df.index.year
        df['trend']     = np.arange(len(df))
        df['lag_1']     = df[self.col].shift(1)
        df['lag_2']     = df[self.col].shift(2)
        df['lag_3']     = df[self.col].shift(3)
        df['roll_3']    = df[self.col].shift(1).rolling(3).mean()
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['momentum']  = df[self.col].diff(1)
        df.dropna(inplace=True)
        return df

In [9]:
artifact   = joblib.load("price_model.joblib")
preprocess = artifact['preprocess']
model      = artifact['model']
 